# argentina.bancos — Pruebas interactivas

Recorrido del módulo `argentina.bancos`.

Funciones para limpiar, validar y formatear identificadores bancarios argentinos: CBU, CVU y alias. Solo stdlib — sin red, sin APIs externas, sin scraping.

## 1. Setup

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")
print(f"bancos en tabla: {len(arg.bancos.BANCOS)}")

argentina v0.0.23
bancos en tabla: 14


## 2. limpiar_cbu

Saca cualquier separador (espacios, guiones, puntos) y deja sólo dígitos.

In [2]:
for v in [
    "0170 0001 4000 0001 2345 67",
    "0170-0001-4000-0001-2345-67",
    "0170000140000001234567",
    None,
    "",
    "abc",
]:
    print(f"{v!r:36} → {arg.bancos.limpiar_cbu(v)!r}")

'0170 0001 4000 0001 2345 67'        → '0170000140000001234567'
'0170-0001-4000-0001-2345-67'        → '0170000140000001234567'
'0170000140000001234567'             → '0170000140000001234567'
None                                 → None
''                                   → None
'abc'                                → None


## 3. validar_cbu

Chequea **dos dígitos verificadores reales**:
- bloque 1: dígitos `[0:7]` con DV en posición `7`, pesos `7 1 3 9 7 1 3`
- bloque 2: dígitos `[8:21]` con DV en posición `21`, pesos `3 9 7 1 3 9 7 1 3 9 7 1 3`

Cada DV se calcula como `(10 - suma_pesada % 10) % 10`.

In [3]:
# CBU válido (todos los DV correctos)
arg.bancos.validar_cbu("2850590940090418135201")

True

In [4]:
# Cambiar el último dígito rompe el segundo DV
arg.bancos.validar_cbu("2850590940090418135202")

False

In [5]:
# Acepta separadores (los limpia primero)
arg.bancos.validar_cbu("2850 5909 4009 0418 1352 01")

True

In [6]:
# Casos borde
print(arg.bancos.validar_cbu(None))
print(arg.bancos.validar_cbu(""))
print(arg.bancos.validar_cbu("123"))                          # corto
print(arg.bancos.validar_cbu("28505909400904181352011234"))   # largo

False
False
False
False


## 4. formatear_cbu

Devuelve `XXXXXXXX-XXXXXXXXXXXXXX` (8 dígitos del bloque 1 con su DV, guion, 14 dígitos del bloque 2 con su DV).

In [7]:
for v in [
    "2850590940090418135201",
    "2850 5909 4009 0418 1352 01",
    "123",          # corto
    None,
]:
    print(f"{v!r:30} → {arg.bancos.formatear_cbu(v)!r}")

'2850590940090418135201'       → '28505909-40090418135201'
'2850 5909 4009 0418 1352 01'  → '28505909-40090418135201'
'123'                          → None
None                           → None


## 5. codigo_banco_cbu / banco_por_cbu

Los **primeros 3 dígitos del CBU** son el código del banco (registro BCRA). `banco_por_cbu` los resuelve contra `arg.bancos.BANCOS` (tabla mínima embebida — extensible).

In [8]:
ejemplos = [
    "0170099120000067797370",   # 017 → BBVA
    "2850590940090418135201",   # 285 → Macro
    "0110000100000000000004",   # 011 → Nación
    "3100000000000000000099",   # 310 → Mercado Pago
    "5500000000000000000099",   # prefijo no listado
    None,
]
for v in ejemplos:
    print(f"{v!s:24} código={arg.bancos.codigo_banco_cbu(v)!r:8} banco={arg.bancos.banco_por_cbu(v)!r}")

0170099120000067797370   código='017'    banco='BBVA Argentina'
2850590940090418135201   código='285'    banco='Banco Macro'
0110000100000000000004   código='011'    banco='Banco Nación'
3100000000000000000099   código='310'    banco='Mercado Pago'
5500000000000000000099   código='550'    banco=None
None                     código=None     banco=None


In [9]:
# Tabla completa (mínima inicial — extensible)
for codigo, nombre in sorted(arg.bancos.BANCOS.items()):
    print(f"  {codigo}  {nombre}")

  007  Banco de Galicia
  011  Banco Nación
  014  Banco Provincia
  017  BBVA Argentina
  020  Banco de la Nación Argentina
  027  Banco Supervielle
  029  Banco Ciudad
  034  Patagonia
  044  Santander Argentina
  072  Banco Santander
  093  Banco Chubut
  143  Brubank
  285  Banco Macro
  310  Mercado Pago


In [10]:
# Es un dict mutable: si te falta un banco lo podés agregar en runtime
arg.bancos.BANCOS["305"] = "Banco Hipotecario"
arg.bancos.banco_por_cbu("3050000000000000000099")

'Banco Hipotecario'

## 6. limpiar_alias / validar_alias

Reglas básicas: 6–20 caracteres, solo letras/números/`.`/`-`, todo en mayúsculas.

In [11]:
for v in [
    " Mi.Alias.CBU ",
    "mi-alias-cbu",
    "  ALIAS  CON  ESPACIOS  ",
    None,
    "",
]:
    print(f"{v!r:30} → {arg.bancos.limpiar_alias(v)!r}")

' Mi.Alias.CBU '               → 'MI.ALIAS.CBU'
'mi-alias-cbu'                 → 'MI-ALIAS-CBU'
'  ALIAS  CON  ESPACIOS  '     → 'ALIASCONESPACIOS'
None                           → None
''                             → None


In [12]:
for v in [
    "MI.ALIAS.CBU",          # ok
    "mi-alias-cbu",          # ok (se normaliza a MAYÚS)
    "alias",                  # corto (5)
    "a" * 21,                 # largo
    "alias!cbu",              # caracter inválido
    "alias con espacios",     # alias normalizado queda 'ALIASCONESPACIOS' → válido
    None,
]:
    print(f"{v!r:30} → {arg.bancos.validar_alias(v)}")

'MI.ALIAS.CBU'                 → True
'mi-alias-cbu'                 → True
'alias'                        → False
'aaaaaaaaaaaaaaaaaaaaa'        → False
'alias!cbu'                    → False
'alias con espacios'           → True
None                           → False


## 7. validar_cvu

Por ahora chequeo mínimo: 22 dígitos. Las billeteras virtuales (Mercado Pago, Ualá, Naranja X, etc.) usan CVU con el mismo largo del CBU pero conviven con esquemas distintos por proveedor — extender la validación cuando haga falta.

In [13]:
for v in [
    "0000003100000000000001",   # 22 dígitos
    "123",
    None,
    "abc",
]:
    print(f"{v!r:30} → {arg.bancos.validar_cvu(v)}")

'0000003100000000000001'       → True
'123'                          → False
None                           → False
'abc'                          → False


## 8. Combinando todo

Pipeline típico: una fila cruda con CBU/alias venido de un padrón. Limpiar, validar, identificar banco y formatear.

In [14]:
registros = [
    {"cbu": "2850 5909 4009 0418 1352 01", "alias": " Mi.Alias.CBU "},
    {"cbu": "0110000100000000000004",      "alias": "alias-banco-nacion"},
    {"cbu": "2850590940090418135202",      "alias": "abc"},                # CBU inválido + alias corto
    {"cbu": None,                            "alias": "alias!cbu"},          # sin CBU + alias inválido
]

for r in registros:
    print({
        "cbu_limpio":   arg.bancos.limpiar_cbu(r["cbu"]),
        "cbu_valido":   arg.bancos.validar_cbu(r["cbu"]),
        "cbu_formato":  arg.bancos.formatear_cbu(r["cbu"]),
        "banco":        arg.bancos.banco_por_cbu(r["cbu"]),
        "alias":        arg.bancos.limpiar_alias(r["alias"]),
        "alias_valido": arg.bancos.validar_alias(r["alias"]),
    })

{'cbu_limpio': '2850590940090418135201', 'cbu_valido': True, 'cbu_formato': '28505909-40090418135201', 'banco': 'Banco Macro', 'alias': 'MI.ALIAS.CBU', 'alias_valido': True}
{'cbu_limpio': '0110000100000000000004', 'cbu_valido': False, 'cbu_formato': '01100001-00000000000004', 'banco': 'Banco Nación', 'alias': 'ALIAS-BANCO-NACION', 'alias_valido': True}
{'cbu_limpio': '2850590940090418135202', 'cbu_valido': False, 'cbu_formato': '28505909-40090418135202', 'banco': 'Banco Macro', 'alias': 'ABC', 'alias_valido': False}
{'cbu_limpio': None, 'cbu_valido': False, 'cbu_formato': None, 'banco': None, 'alias': 'ALIAS!CBU', 'alias_valido': False}


## 9. Tests automáticos

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_bancos.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`re`). Sin pandas, sin requests, sin APIs externas.
- `validar_cbu` chequea formato + dígitos verificadores reales. **No** chequea contra el padrón del BCRA — un CBU sintácticamente válido puede no existir.
- `BANCOS` es una tabla mínima inicial. Para usos serios conviene completarla con la lista oficial del BCRA (más de 80 entidades). Es un `dict` plano: extensible en runtime.
- `validar_cvu` por ahora sólo chequea largo 22. Cada billetera virtual (MP, Ualá, etc.) usa esquema propio: si en algún momento se quiere distinguir CBU de CVU por prefijo, agregar una función `tipo_cuenta(...)` aparte.
- Reglas de alias: tomé las más comunes (6–20, alfanum + `.`/`-`). Algunos bancos aceptan `_` o `/`; si te interesa otro charset, parametrizar.